# MODIS time series analysis using LSTM

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data.csv")
df.sort_values(['time','pixel'], inplace=True)

In [ ]:
df.info()

In [ ]:
df.head()

Here each pixel represet one location and reprsent 250*250 region, 
we can see there are 437 time observations for each pixel

In [ ]:
df['pixel'].value_counts()

In [ ]:
df['qa'].hist(bins=20)

In [ ]:
subset = df[
            (df['pixel']==1)
             ]

plt.figure(figsize=(15, 6))
scatter = plt.scatter(subset['time'], subset['ndvi'], cmap='viridis', s=5)
plt.colorbar(scatter, label='NDVI')
plt.xlabel('Pixel')
plt.ylabel('NDVI')
plt.title('NDVI vs Pixel')
plt.show()

In [ ]:
df.land_cover

df.isna().sum()

In [ ]:
df = df.dropna()

##  Data exploration

In [ ]:
df.groupby('pixel').count()

### Low quality values

We can see that there are a lot of abnormal values or outliers.
Thus, we need to filter them ased on the QA column. 

There are two strategies to process these low quality or abnormal values

1 to remove them

2 to fill them with mean of neaby locations

### Low quality and abnormla values

In [ ]:
subset = df[
            (df['pixel']==1) &
            (df['qa']>=0) &
            (df['qa']<=1)
             ]

plt.figure(figsize=(15, 6))
scatter = plt.scatter(subset['time'], subset['ndvi'], c=subset['qa'], cmap='viridis', s=5)

plt.colorbar(scatter, label='QA Value')
plt.xlabel('Pixel')
plt.ylabel('NDVI')
plt.title('NDVI vs Pixel with Color Based on QA')
plt.show()

In [ ]:
subset.isna().sum()

In [ ]:
df.qa.hist()

In [ ]:
df['time'] = pd.to_datetime(df['time'])

In [ ]:
df.time

In [ ]:
df = df.sort_values(by=['pixel', 'time'])

In [ ]:
df.head()


In [ ]:
from sklearn.model_selection import train_test_split

features = []

# Function to create sequences of data for LSTM
def create_sequences(df, seq_length=3):
    sequences = []
    targets = []
    for pixel in df['pixel'].unique():
        pixel_data = df[df['pixel'] == pixel]
        for i in range(len(pixel_data) - seq_length):
            seq = pixel_data.iloc[i:i+seq_length].drop(columns=['pixel', 'time', 'lc_time']).values  # Drop non-input features
            target = pixel_data.iloc[i + seq_length]['ndvi']  # The value to predict (e.g., ndvi)
            sequences.append(seq)
            targets.append(target)

    return np.array(sequences), np.array(targets)

# Create sequences
seq_length = 10
X, y = create_sequences(df[0:100000], seq_length)

# Split into train and test sets (make sure the split is done per pixel and chronologically)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
X_train_tensor = torch.Tensor(X_train)  # Shape: (num_samples, seq_length, num_features)
y_train_tensor = torch.Tensor(y_train)  # Shape: (num_samples,)
X_test_tensor = torch.Tensor(X_test)  # Shape: (num_samples, seq_length, num_features)
y_test_tensor = torch.Tensor(y_test)  # Shape: (num_samples,)

X_train_tensor, X_val_tensor, y_train_tensor, y_val_tensor = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.2, shuffle=False
)

#### Write a model's class

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Get the LSTM output (the full sequence) and hidden state (the last output in the sequence)
        lstm_out, (hn, cn) = self.lstm(x)
        # We use the last hidden state to predict the output (for time series prediction)
        out = self.fc(hn[-1])  # Take the last hidden state
        return out

In [ ]:
# Hyperparameters
input_size = X_train.shape[2]  # Number of features
hidden_size = 50  # Number of hidden units in LSTM layer
output_size = 1
num_layers = 1
learning_rate = 0.001
num_epochs = 10

# Instantiate the model
model = LSTMModel(input_size, hidden_size, output_size, num_layers)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

#### Train model

In [ ]:
train_losses = []
val_losses = []

# Training loop
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    # Forward pass (Training)
    y_train_pred = model(X_train_tensor)

    # Compute training loss
    train_loss = criterion(y_train_pred.squeeze(), y_train_tensor)

    # Backward pass and optimization
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val_tensor)
        val_loss = criterion(y_val_pred.squeeze(), y_val_tensor)

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss.item():.4f}, Val Loss: {val_loss.item():.4f}")


#### Plot the training and validation losses

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs+1), train_losses, label="Training Loss", color='blue')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", color='red')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Losses')
plt.legend()
plt.grid(True)
plt.show()

#### Evaluate the model on the test set

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test_tensor)
    test_loss = mean_squared_error(y_test_tensor, y_pred_test.squeeze())
    print(f"Test Mean Squared Error: {test_loss:.4f}")

Tasks:
1) Drop rows based on QA flag and fill empty rows (pandas.DataFrame.bfill)
2) Perform one-hot-encoding for categorical features
3) Train a model with multiple output targets (multiple target values in each sequence)
4) Modify the model
5) Make comparison of different experiments
6) Write text report